This notebook introduces fundamental PyTorch commands and their application in computer vision. It covers essential topics such as creating and manipulating PyTorch tensors, loading and preprocessing image datasets (like MNIST and CIFAR10), and applying transformations for normalization and augmentation. By the end of this notebook you should be familiar with basic PyTorch operations, dataset handling and common image preprocessing techniques in the context of computer vision.

# 1.2 PyTorch for Computer Vision

In this note, we will address some of the basic commands that you will need from PyTorch in general and in computer vision in particular.

<!---
- [Basic PyTorch commands](#Basic-PyTorch-commands)
- [Computer vision data sets](#Computer-vision-data-sets)
- [Transforms](#Transforms)
- [Exercises](#Exercises)
-->

First, we import the required libraries. These should all be available on Google Colab.

In [ ]:
import torch # PyTorch
import torchvision # Computer vision
# Data sets and transforms
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
# Other libraries
import matplotlib.pyplot as plt
import numpy as np

You can, e.g., check your PyTorch

In [ ]:
torch.__version__

## Basic PyTorch commands

PyTorch has its own objects: PyTorch tensors. All data will have to be put in this format.

In [ ]:
scalar = torch.tensor(1)
vector = torch.tensor([1, 2])

If you already have a tensor object, you can print different properties.

In [ ]:
print(vector.ndim, vector.shape,vector.dtype)

Many of the commands (such as the method creating vectors with ones in each entry) mirror the corresponding commands in numpy.

In [ ]:
t0 = torch.zeros(size=(3,1))
print(t0)
t1 = torch.ones(size=(3,1))
print(t1)

Just like in numpy you can, for instance, easily perform element-wise multiplication, multiply a tensor with a scalar or determine the index of the entry with the maximum or minimum value. But not everything is the same.

In [ ]:
print(10*vector)
print(vector*vector)
print(vector.argmax())
print(vector.argmin())

When creating so-called batches of data in later notebooks, we will sometimes need to be able to add a dimension to a tensor. We thus keep the content of the data but add an empty dimension along the dimension dim.

In [ ]:
torch.unsqueeze(vector,dim=0)

You can also turn PyTorch tensors into numpy objects

In [ ]:
vector.numpy()

Or numpy objects into tensors.

In [ ]:
array = np.arange(1,3)
torch.from_numpy(array)

## Computer vision data sets

In this course, we will dive into Deep Learning for computer vision, i.e., we will be working with image data. Torchvision provides a long list of datasets that are commonly used for this purpose.

In [ ]:
# Available datasets
print(datasets.__all__)

To dowload and import the training set from the MNIST data set, we use the command

In [ ]:
trainset_mnist_0 = datasets.MNIST(root='./data', train=True, download=True)

Note that not all datasets listed above can be easily downloaded with one line of code.

We can plot individual images from this data set using matplotlib. Each image is a PyTorch tensor object.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img = trainset_mnist_0.data[i]
    print(type(img))
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')

Let's try a different dataset.

In [ ]:
trainset_cifar10_0 = datasets.CIFAR10(root='./data', train=True, download=True)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img = trainset_cifar10_0.data[i]
    axes[i].imshow(img)
    axes[i].axis('off')

## Transforms

Before we can pass the data to our Deep Learning model, we will often have to transform it to ensure that it has the correct format. Maybe the images are too small or too large compared to the images on which we have trained our model. It is also common practice to normalise the images before passing them to the Deep Learning model, as this approach is beneficial for convergence, stability during training and the generalisation of the network.

Let's define a transform for normalisation.

In [ ]:
# MNIST is black and white. So, we only need one color channel. Other data sets have three
means = (0.5)
stds = (0.5)

# Define the data transform including normalization
transform_mnist_0 = transforms.Compose([
    transforms.ToTensor(),  # convert to PyTorch tensor
    transforms.Normalize(mean=means, std=stds)  # normalize images output[channel] = (input[channel] - mean[channel]) / std[channel]
])

# In contrast, CIFAR10 provides pictures with three channels (RGB).

means10 = (0.5,0.5,0.5)
stds10 = (0.5,0.5,0.5)

# Define the data transform including normalization
transform_cifar10_0 = transforms.Compose([
    transforms.ToTensor(),  # Converts the image to a PyTorch tensor
    transforms.Normalize(means10, stds10),  # Normalize to a range of -1 to 1 for three channels
])

If we had defined the transform before we loaded the data, we could have applied the transform to the data right away. However, we can still do so retrospectively.

In [ ]:
# Apply the transform to the dataset
trainset_cifar10_0.transform = transform_cifar10_0

We can divide the entire dataset into smaller batches, shuffle it and provide an iterable interface for retrieving batches during the training process using the DataLoader class.

In [ ]:
train_loader_cifar10 = DataLoader(trainset_cifar10_0, batch_size=32, shuffle=True)

If you plot the image, you can now see that the colours are a bit off because we normalised the image.

In [ ]:
# Get the first batch from the data loader
dataiter = iter(train_loader_cifar10)
images, labels = next(dataiter)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img = images[i].permute(1, 2, 0)
    axes[i].imshow(img)
    axes[i].axis('off')

We can also use transforms for image augmentation, i.e., we scale, translate or rotate the images to create new images. By doing so, we reduce overfitting and introduce scale and orientation invariance.

In [ ]:
transform_mnist_1 = transforms.Compose([
    transforms.RandomHorizontalFlip(), # Randomflip
    transforms.RandomRotation(degrees=(-90, 90)),  # Random rotation between -90 and 90 degrees
    transforms.ToTensor(),  # convert to PyTorch tensor
    transforms.Normalize(mean=means, std=stds)  # normalize images
])

# Apply the transform to the dataset
trainset_mnist_0.transform = transform_mnist_1
train_loader_mnist = DataLoader(trainset_mnist_0, batch_size=32, shuffle=True)

In [ ]:
# Get the first batch from the data loader
dataiter = iter(train_loader_mnist)
images, labels = next(dataiter)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img = images[i].permute(1, 2, 0)
    axes[i].imshow(img, cmap="gray")
    axes[i].axis('off')
    axes[i].set_title(labels[i].item())
